### Use this analysis to:
1. **Identify Sales Trends**: Monitor monthly and seasonal patterns
2. **Optimize Product Mix**: Focus on high-margin, top-performing products
3. **Improve Operations**: Track delivery performance and identify bottlenecks
4. **Customer Segmentation**: Analyze geographic and demographic patterns
5. **Strategic Planning**: Use YoY/MoM growth insights for forecasting

In [ ]:
import pandas as pd
import numpy as np
import pyodbc
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)
import plotly.io as pio
pio.templates.default = "plotly_white"

## Database config

In [ ]:
# Run below queries in ssms
# SELECT @@SERVERNAME AS ServerName;
# SELECT DB_NAME() AS CurrentDatabase;

SERVER = 'DESKTOP-B2N506K\SQLEXPRESS02'
DATABASE = 'DataWarehouse'

connection_string = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={SERVER};DATABASE={DATABASE};Trusted_Connection=yes;'

print(f"Connection configured for: {SERVER}")
print(f"Database: {DATABASE}")

In [ ]:
try:
    conn = pyodbc.connect(connection_string)
    print("Database connected")
    conn.close()
except Exception as e:
    print(f"Connection failed: {e}")
    print("SQL Server must be running")
    print("Check db config")

## Execute SQL query and return as df

In [ ]:
def query_to_dataframe(query, conn_string):
    try:
        conn = pyodbc.connect(conn_string)
        df = pd.read_sql(query, conn)
        conn.close()
        return df
    except Exception as e:
        print(f"Query execution failed: {e}")
        return None

## Get sales data based on customer profile

In [ ]:
sales_query = """
SELECT 
    f.order_number,
    f.order_date,
    f.ship_date,
    f.due_date,
    f.sales_amount,
    f.quantity,
    f.price,
    pr.product_name,
    pr.category,
    pr.subcategory,
    pr.product_line,
    pr.product_cost,
    cu.customer_number,
    cu.first_name,
    cu.last_name,
    cu.country,
    cu.gender,
    cu.marital_status
FROM gold.fact_sales f
LEFT JOIN gold.dim_products pr ON f.product_key = pr.product_key
LEFT JOIN gold.dim_customers cu ON f.customer_key = cu.customer_key
ORDER BY cu.customer_number
"""

print("Load Sales >> product info >> customer profile")
df_sales = query_to_dataframe(sales_query, connection_string)

if df_sales is not None:
    print(f"Total {len(df_sales):,} sales records")
    print(f"Range: {pd.to_datetime(df_sales['order_date']).min().date()} to {pd.to_datetime(df_sales['order_date']).max().date()}")
    print(f"Total revenue: ${df_sales['sales_amount'].sum():,.2f}")
else:
    print("Failed to load sales data")

In [ ]:
df_sales.head(10)

In [ ]:
print(f"Total Records: {len(df_sales):,}")
print(f"Date Range: {pd.to_datetime(df_sales['order_date']).min().date()} to {pd.to_datetime(df_sales['order_date']).max().date()}")
print(f"Unique Customers: {df_sales['customer_number'].nunique():,}")
print(f"Unique Products: {df_sales['product_name'].nunique():,}")
print(f"Unique Countries: {df_sales['country'].nunique():,}")
df_sales.info()

## Feature Engineering

In [ ]:
df = df_sales.copy()

# Cast again to ensure consistency
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])
df['due_date'] = pd.to_datetime(df['due_date'])

# Extract date components
df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['quarter'] = df['order_date'].dt.quarter
df['day_of_week'] = df['order_date'].dt.day_name()
df['month_name'] = df['order_date'].dt.strftime('%Y-%m')
df['year_month'] = df['order_date'].dt.to_period('M')

# na = 0 to prevent errors in calculation
df['profit'] = df['sales_amount'] - (df['product_cost'] * df['quantity'])
df['profit_margin'] = (df['profit'] / df['sales_amount'] * 100).fillna(0)

df['days_to_ship'] = (df['ship_date'] - df['order_date']).dt.days
df['on_time_delivery'] = df['ship_date'] <= df['due_date']

print("Data preparation complete")

## Sales Performance

In [ ]:
total_revenue = df['sales_amount'].sum()
total_profit = df['profit'].sum()
total_orders = df['order_number'].nunique()
total_quantity = df['quantity'].sum()
avg_order_value = df.groupby('order_number')['sales_amount'].sum().mean()
avg_profit_margin = (total_profit / total_revenue * 100) if total_revenue > 0 else 0
unique_customers = df['customer_number'].nunique()
unique_products = df['product_name'].nunique()


print(">> SALES PERFORMANCE DASHBOARD")
print(f"""\n
REVENUE METRICS
Total Revenue:        ${total_revenue:,.2f}
Total Profit:         ${total_profit:,.2f}
Profit Margin:        {avg_profit_margin:.2f}%

ORDER METRICS
Total Orders:         {total_orders:,}
Total Units Sold:     {total_quantity:,.0f}
Average Order Value:  ${avg_order_value:,.2f}

CUSTOMER & PRODUCT METRICS
Unique Customers:     {unique_customers:,}
Unique Products:      {unique_products:,}
Revenue per Customer: ${total_revenue/unique_customers:,.2f}
""")

## Trends Over Time

In [ ]:
monthly_sales = df.groupby('month_name').agg({
    'sales_amount': 'sum',
    'profit': 'sum',
    'order_number': 'nunique',
    'quantity': 'sum'
}).reset_index()

monthly_sales.columns = ['Month', 'Revenue', 'Profit', 'Orders', 'Units_Sold']
monthly_sales['Profit_Margin_%'] = (monthly_sales['Profit'] / monthly_sales['Revenue'] * 100).round(2)


fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Monthly Revenue & Profit Trend', 'Monthly Orders & Units Sold'),
    vertical_spacing=0.12,
    specs=[[{"secondary_y": True}], [{"secondary_y": True}]]
)

fig.add_trace(
    go.Scatter(x=monthly_sales['Month'], y=monthly_sales['Revenue'], 
               name='Revenue', mode='lines+markers', line=dict(color='#1f77b4', width=3)),
    row=1, col=1, secondary_y=False
)
fig.add_trace(
    go.Scatter(x=monthly_sales['Month'], y=monthly_sales['Profit'], 
               name='Profit', mode='lines+markers', line=dict(color='#2ca02c', width=3)),
    row=1, col=1, secondary_y=False
)
fig.add_trace(
    go.Scatter(x=monthly_sales['Month'], y=monthly_sales['Profit_Margin_%'], 
               name='Profit Margin %', mode='lines', line=dict(color='#ff7f0e', width=2, dash='dash')),
    row=1, col=1, secondary_y=True
)

fig.add_trace(
    go.Scatter(x=monthly_sales['Month'], y=monthly_sales['Orders'], 
               name='Orders', mode='lines+markers', line=dict(color='#d62728', width=3)),
    row=2, col=1, secondary_y=False
)
fig.add_trace(
    go.Scatter(x=monthly_sales['Month'], y=monthly_sales['Units_Sold'], 
               name='Units Sold', mode='lines+markers', line=dict(color='#9467bd', width=3)),
    row=2, col=1, secondary_y=True
)

fig.update_yaxes(title_text="Revenue / Profit ($)", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Profit Margin (%)", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="Orders", row=2, col=1, secondary_y=False)
fig.update_yaxes(title_text="Units Sold", row=2, col=1, secondary_y=True)

fig.update_layout(height=700, title_text="Sales Performance Over Time", showlegend=True)
fig.show()

print("\nMonthly Sales Summary:")
monthly_sales

In [ ]:
daily_sales = df.groupby('order_date').agg({
    'sales_amount': 'sum',
    'order_number': 'nunique'
}).reset_index()

fig = px.line(daily_sales, x='order_date', y='sales_amount', 
              title='Daily Sales Revenue Trend',
              labels={'order_date': 'Date', 'sales_amount': 'Revenue ($)'})
fig.update_traces(line_color='#1f77b4', line_width=2)
fig.update_layout(height=400)
fig.show()

## YoY and MoM

In [ ]:
yearly_sales = df.groupby('year').agg({
    'sales_amount': 'sum',
    'profit': 'sum',
    'order_number': 'nunique',
    'customer_number': 'nunique'
}).reset_index()

yearly_sales.columns = ['Year', 'Revenue', 'Profit', 'Orders', 'Customers']
yearly_sales['YoY_Revenue_Growth_%'] = yearly_sales['Revenue'].pct_change() * 100
yearly_sales['YoY_Order_Growth_%'] = yearly_sales['Orders'].pct_change() * 100

print("YoY Performance:")
print(yearly_sales.to_string(index=False))


fig = make_subplots(rows=1, cols=2, subplot_titles=('Yearly Revenue', 'YoY Growth %'))

fig.add_trace(
    go.Bar(x=yearly_sales['Year'], y=yearly_sales['Revenue'], name='Revenue',
           marker_color='#1f77b4', text=yearly_sales['Revenue'].apply(lambda x: f'${x:,.0f}')),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=yearly_sales['Year'], y=yearly_sales['YoY_Revenue_Growth_%'], name='YoY Growth %',
           marker_color='#2ca02c', text=yearly_sales['YoY_Revenue_Growth_%'].apply(lambda x: f'{x:.1f}%' if pd.notna(x) else 'N/A')),
    row=1, col=2
)

fig.update_traces(textposition='outside')
fig.update_layout(height=400, showlegend=False, title_text="YoY Analysis")
fig.show()

In [ ]:
monthly_sales_sorted = monthly_sales.sort_values('Month')
monthly_sales_sorted['MoM_Revenue_Growth_%'] = monthly_sales_sorted['Revenue'].pct_change() * 100
monthly_sales_sorted['MoM_Orders_Growth_%'] = monthly_sales_sorted['Orders'].pct_change() * 100

fig = go.Figure()
fig.add_trace(go.Bar(
    x=monthly_sales_sorted['Month'],
    y=monthly_sales_sorted['MoM_Revenue_Growth_%'],
    name='MoM Revenue Growth %',
    marker_color=monthly_sales_sorted['MoM_Revenue_Growth_%'].apply(lambda x: '#2ca02c' if x >= 0 else '#d62728')
))

fig.update_layout(
    title='MoM Revenue Growth %',
    xaxis_title='Month',
    yaxis_title='Growth %',
    height=400
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.show()

print("\nMoM Growth:")
print(monthly_sales_sorted[['Month', 'Revenue', 'MoM_Revenue_Growth_%']].tail(12).to_string(index=False))

## Products Analysis

In [ ]:
top_products = df.groupby('product_name').agg({
    'sales_amount': 'sum',
    'profit': 'sum',
    'quantity': 'sum',
    'order_number': 'nunique'
}).reset_index()

top_products.columns = ['Product', 'Revenue', 'Profit', 'Units_Sold', 'Orders']
top_products['Profit_Margin_%'] = (top_products['Profit'] / top_products['Revenue'] * 100).round(2)
top_products = top_products.sort_values('Revenue', ascending=False).head(10)

fig = px.bar(top_products, x='Revenue', y='Product', orientation='h',
             title='Top 10 Products by Revenue',
             labels={'Revenue': 'Revenue ($)', 'Product': ''},
             color='Profit_Margin_%',
             color_continuous_scale='RdYlGn',
             text='Revenue')

fig.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig.update_layout(height=500, yaxis={'categoryorder':'total ascending'})
fig.show()

print("\nTop 10 Products:")
print(top_products.to_string(index=False))

## Category analysis

In [ ]:
category_sales = df.groupby('category').agg({
    'sales_amount': 'sum',
    'profit': 'sum',
    'quantity': 'sum',
    'order_number': 'nunique'
}).reset_index()

category_sales.columns = ['Category', 'Revenue', 'Profit', 'Units_Sold', 'Orders']
category_sales['Profit_Margin_%'] = (category_sales['Profit'] / category_sales['Revenue'] * 100).round(2)
category_sales = category_sales.sort_values('Revenue', ascending=False)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Revenue by Category', 'Profit by Category'),
    specs=[[{'type':'pie'}, {'type':'pie'}]]
)

fig.add_trace(
    go.Pie(labels=category_sales['Category'], values=category_sales['Revenue'], 
           name='Revenue', hole=0.3),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=category_sales['Category'], values=category_sales['Profit'], 
           name='Profit', hole=0.3),
    row=1, col=2
)

fig.update_layout(height=400, title_text="Category Distribution")
fig.show()

print("\nCategory analysis:")
print(category_sales.to_string(index=False))

In [ ]:
subcategory_sales = df.groupby(['category', 'subcategory']).agg({
    'sales_amount': 'sum',
    'profit': 'sum'
}).reset_index()

subcategory_sales.columns = ['Category', 'Subcategory', 'Revenue', 'Profit']
subcategory_sales = subcategory_sales.sort_values('Revenue', ascending=False).head(15)

fig = px.treemap(subcategory_sales, 
                 path=['Category', 'Subcategory'], 
                 values='Revenue',
                 color='Profit',
                 color_continuous_scale='RdYlGn',
                 title='Category & Subcategory Revenue Breakdown (Top 15)')

fig.update_layout(height=500)
fig.show()

## Seasonal analysis

In [ ]:
quarterly_sales = df.groupby(['year', 'quarter']).agg({
    'sales_amount': 'sum',
    'order_number': 'nunique'
}).reset_index()

quarterly_sales['Year_Quarter'] = quarterly_sales['year'].astype(str) + '-Q' + quarterly_sales['quarter'].astype(str)

fig = px.bar(quarterly_sales, x='Year_Quarter', y='sales_amount',
             title='Quarterly Sales Performance',
             labels={'sales_amount': 'Revenue ($)', 'Year_Quarter': 'Quarter'},
             color='sales_amount',
             color_continuous_scale='Blues')

fig.update_layout(height=400)
fig.show()

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

dow_sales = df.groupby('day_of_week').agg({
    'sales_amount': 'sum',
    'order_number': 'nunique'
}).reset_index()

dow_sales.columns = ['Day', 'Revenue', 'Orders']
dow_sales['Day'] = pd.Categorical(dow_sales['Day'], categories=day_order, ordered=True)
dow_sales = dow_sales.sort_values('Day')


fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(x=dow_sales['Day'], y=dow_sales['Revenue'], name='Revenue', marker_color='#1f77b4'),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(x=dow_sales['Day'], y=dow_sales['Orders'], name='Orders', 
               mode='lines+markers', marker_color='#ff7f0e', line=dict(width=3)),
    secondary_y=True
)

fig.update_layout(title='Sales by Day of Week', height=400)
fig.update_yaxes(title_text="Revenue ($)", secondary_y=False)
fig.update_yaxes(title_text="Number of Orders", secondary_y=True)
fig.show()

print("\nSales by Day of Week:")
print(dow_sales.to_string(index=False))

## Geographic analysis

In [ ]:
country_sales = df.groupby('country').agg({
    'sales_amount': 'sum',
    'profit': 'sum',
    'order_number': 'nunique',
    'customer_number': 'nunique'
}).reset_index()

country_sales.columns = ['Country', 'Revenue', 'Profit', 'Orders', 'Customers']
country_sales = country_sales.sort_values('Revenue', ascending=False)

fig = px.bar(country_sales, x='Country', y='Revenue',
             title='Revenue by Country',
             labels={'Revenue': 'Revenue ($)'},
             color='Revenue',
             color_continuous_scale='Viridis')

fig.update_layout(height=400, xaxis={'categoryorder':'total descending'})
fig.show()

print("\nTop Countries by Revenue:")
print(country_sales.head(10).to_string(index=False))

## Price vs Quantity

In [ ]:
fig = px.scatter(df, x='price', y='quantity', 
                 color='category',
                 size='sales_amount',
                 hover_data=['product_name', 'sales_amount'],
                 title='Price vs Quantity Analysis',
                 labels={'price': 'Unit Price ($)', 'quantity': 'Quantity Sold'},
                 opacity=0.6)

fig.update_layout(height=500)
fig.show()

## Profit margin analysis

In [ ]:
fig = px.histogram(df, x='profit_margin', 
                   title='Profit Margin Distribution',
                   labels={'profit_margin': 'Profit Margin (%)', 'count': 'Frequency'},
                   nbins=50,
                   color_discrete_sequence=['#2ca02c'])

fig.update_layout(height=400)
fig.show()


print(f"Profit margin analysis:")
print(f"Mean: {df['profit_margin'].mean():.2f}%")
print(f"Median: {df['profit_margin'].median():.2f}%")
print(f"Std Dev: {df['profit_margin'].std():.2f}%")
print(f"Min: {df['profit_margin'].min():.2f}%")
print(f"Max: {df['profit_margin'].max():.2f}%")

## Delivery analysis

In [ ]:
on_time_rate = df['on_time_delivery'].mean() * 100
avg_delivery_days = df['days_to_ship'].mean()

print(f"Delivery Performance Metrics:")
print(f"On-Time Delivery Rate: {on_time_rate:.2f}%")
print(f"Average Days to Ship: {avg_delivery_days:.1f} days")

delivery_monthly = df.groupby('month_name').agg({
    'on_time_delivery': 'mean',
    'days_to_ship': 'mean'
}).reset_index()

delivery_monthly['on_time_delivery'] = delivery_monthly['on_time_delivery'] * 100

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=delivery_monthly['month_name'], y=delivery_monthly['on_time_delivery'],
               name='On-Time %', mode='lines+markers', line=dict(color='#2ca02c', width=3)),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(x=delivery_monthly['month_name'], y=delivery_monthly['days_to_ship'],
               name='Avg Days to Ship', mode='lines+markers', line=dict(color='#ff7f0e', width=3)),
    secondary_y=True
)

fig.update_layout(title='Delivery analysis Over Time', height=400)
fig.update_yaxes(title_text="Timely delivery %", secondary_y=False)
fig.update_yaxes(title_text="Average shipping time", secondary_y=True)
fig.show()

## Summary

In [ ]:
summary_report = {
    'Total Revenue': f"${total_revenue:,.2f}",
    'Total Profit': f"${total_profit:,.2f}",
    'Overall Profit Margin': f"{avg_profit_margin:.2f}%",
    'Total Orders': f"{total_orders:,}",
    'Total Units Sold': f"{total_quantity:,.0f}",
    'Average Order Value': f"${avg_order_value:,.2f}",
    'Unique Customers': f"{unique_customers:,}",
    'Unique Products': f"{unique_products:,}",
    'On-Time Delivery Rate': f"{on_time_rate:.2f}%",
    'Average Delivery Days': f"{avg_delivery_days:.1f}",
    'Date Range': f"{df['order_date'].min().date()} to {df['order_date'].max().date()}"
}

summary_df = pd.DataFrame(list(summary_report.items()), columns=['Metric', 'Value'])
print("\n" + "="*60)
print(" "*15 + "EXECUTIVE SUMMARY REPORT")
print("="*60)
print(summary_df.to_string(index=False))
print("="*60)


# summary_df.to_csv('sales_performance_summary.csv', index=False)
# monthly_sales.to_csv('monthly_sales_report.csv', index=False)
# category_sales.to_csv('category_performance.csv', index=False)